# P16 · Construcción de factores financieros
**QuantLab | Módulo 1 | Lecciones 1.26 y 1.30**

## 1. Objetivo
Construir carteras long-short a partir de características observables antes de invertir, medir sus retornos y costes, y distinguir un factor construido de una exposición estimada por regresión (P12). Incluye un repaso programable de derivados.

Pregunta: ¿los activos con una característica determinada obtienen retornos distintos de los activos con la característica opuesta? Los datos iniciales son artificiales; no demuestran una prima de riesgo real. Los ejercicios están resueltos en la sección 12, y el E7 repite el análisis con datos reales (Ken French y Yahoo Finance).

Las funciones están en **`src/quantlab/p16/factor_construction.py`** y sus pruebas en **`tests/test_factor_construction.py`** (`pytest` desde la raíz del proyecto).

## 2. Configuración
Dependencias: numpy, pandas, matplotlib, statsmodels (incluidas en `requirements.txt`; el E7 usa además requests y yfinance).

Trabajamos con retornos mensuales simples decimales, todos en la misma moneda, y una posición long de nominal 1 y short de nominal 1 por unidad de capital de referencia. El spread long-short no es el retorno de una cartera spot convencional: hay exposición bruta 2x, garantías y préstamo no modelados completamente.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pathlib import Path
import inspect
import sys

try:
    import quantlab
except ImportError:  # sin `pip install -e .`: buscamos src/ subiendo desde el cuaderno
    src = next((d / "src" for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "quantlab").exists()), None)
    if src is None:
        raise
    sys.path.insert(0, str(src))
from quantlab.p16.factor_construction import (CRYPTO_TICKERS, FF_FILES, SCORES, add_scores, annualized_basis,
                                              breakeven_cost_bps, build_factor, crypto_monthly_panel, derivative_pnl,
                                              download_daily, download_ff, factor_table, fit_model, long_short_weights,
                                              manual_long_short, model_summary, momentum_12_2, monthly_returns,
                                              multiple_testing, perp_carry_score, period_summary, quantile_comparison,
                                              read_ff_monthly, spanning_alpha, summarize, swap_net_flow,
                                              synthetic_panel, validate_panel)

DATA_MODE = 'synthetic'  # synthetic | csv
PANEL_CSV = 'asset_monthly_panel.csv'
FF3_CSV = None  # fichero CSV oficial mensual de Ken French, descomprimido
FF5_CSV = None
STRATEGY_CSV = None  # month,strategy_return (decimal; misma moneda que FF)
SEED = 42
QUANTILE = 0.30
COST_BPS = 15  # coste hipotético por unidad de nominal negociado, por lado
TRAIN_SHARE = 0.65
RF_MONTHLY = 0.002  # solo ejemplo sintético
plt.rcParams.update({'figure.figsize':(9,4), 'axes.spines.top':False,
                     'axes.spines.right':False, 'axes.grid':True, 'grid.alpha':.2,
                     'figure.dpi':120})
assert 0 < QUANTILE <= .5 and COST_BPS >= 0

## 3. Qué es un factor y cómo se construye
Un **score** es una característica disponible en la fecha de decisión. Un **factor return** es el retorno de una cartera definida por esa característica. Para este laboratorio empleamos una ordenación univariante sencilla, equiponderada, no la metodología exacta de Fama-French.

| Factor | Score alto favorecido | Ejemplo |
|---|---|---|
| Size | Menor capitalización | $-\log(cap)$ |
| Value | Mayor book-to-market | Patrimonio contable / capitalización |
| Quality | Mayor rentabilidad contable | ROA (proxy) |
| Momentum | Mayor retorno pasado | Meses t-12 a t-2 |
| Low Vol | Menor volatilidad pasada | Desviación típica 6 meses previos |
| Carry | Mayor rendimiento de mantenimiento esperado | Proxy suministrado, según instrumento |
| Liquidity | Mayor iliquidez | Proxy de iliquidez (prima hipotética) |
| Investment | Menor crecimiento de activos | Proxy de inversión conservadora |

El factor mercado de esta práctica es el retorno equiponderado menos el tipo libre de riesgo. Los scores de Value y Quality pertenecen al universo de acciones; no los traslades directamente a BTC. Para perpetuos, el carry depende de la financiación y del basis, no es un dividendo.

## 4. Datos y disponibilidad temporal
Una fila representa un activo y un mes de retorno. `month` es el primer día del mes de tenencia; `available_at` indica cuándo se conocían las características. Para operar, los fundamentales deben estar disponibles estrictamente antes de `month`. `asset_return` es el retorno total realizado durante el mes, no una entrada en la decisión.

El CSV real debe contener `month, asset, available_at, market_cap, book_to_market, roa, carry_score, illiquidity, asset_growth, asset_return, rf`. Retornos y RF en decimal, identificadores de activos estables e incluir bajas de cotización y retornos de exclusión cuando proceda. La fecha declarada no demuestra por sí sola la ausencia de revisiones o sesgo de supervivencia. No se rellenan retornos ausentes ni se eliminan posiciones perdedoras a posteriori.

En la simulación, **solo value (+) y size (las pequeñas ganan) tienen una prima programada**; el resto de características no afecta a los retornos. Size, value y quality evolucionan cada mes; carry, iliquidez e inversión son constantes por activo.

<div style="background-color:rgba(240,140,0,0.14); border-left:6px solid #f08c00; padding:12px 16px; border-radius:6px;">

<b style="color:#f08c00;">🔎 FASE DE VALIDACIÓN · ¿el panel se puede usar para decidir?</b>

`validate_panel` se detiene con un error (no corrige nada) si:

- `month` no es el primer día del mes, o alguna característica tiene `available_at` ≥ `month` (no se conocía al decidir).
- Hay filas activo-mes duplicadas, capitalizaciones ≤ 0 o retornos inferiores a −100 %.
- El RF no es único dentro de un mes.

Los retornos ausentes se quedan como NaN: si el activo es elegido, ese mes del factor es desconocido.

</div>

In [ ]:
print(inspect.getsource(synthetic_panel))
print(inspect.getsource(validate_panel))

panel = synthetic_panel(SEED, rf_monthly=RF_MONTHLY) if DATA_MODE=='synthetic' else pd.read_csv(PANEL_CSV)
panel = validate_panel(panel)
print(f'{len(panel):,} filas | {panel.asset.nunique()} activos | {panel.month.nunique()} meses | {DATA_MODE}')
display(panel.head(4))

## 5. Construir scores sin información futura
En el mes t, el Momentum 12-2 utiliza los 11 retornos de t-12 a t-2; Low Vol utiliza t-6 a t-1. Reindexamos el calendario mensual completo: si falta un mes, no lo confundimos con un retorno cero ni con el mes anterior. Las características contables provienen de `available_at` validado.

In [ ]:
print(inspect.getsource(add_scores))

panel, mom, lowvol, all_months = add_scores(panel)
display(panel[['month','asset']+SCORES].tail(4))

## 6. Ordenar activos y formar carteras
En cada mes se ordenan únicamente los activos con score conocido. Compramos el 30 % superior y vendemos el 30 % inferior (redondeo hacia abajo). Los pesos de cada tramo suman +1 y -1. Los empates se resuelven por identificador de activo para reproducibilidad; si todos los scores empatan, no hay señal. Si falta el retorno de una posición seleccionada, la rentabilidad de ese mes es desconocida, no cero.

In [ ]:
print(inspect.getsource(long_short_weights))

sample_month=all_months[12]
sample=panel[panel.month==sample_month]
w=long_short_weights(sample,'value',QUANTILE)
display(pd.DataFrame({'score':sample.set_index('asset').value,'weight':w}).sort_values('score').iloc[np.r_[0:3,-3:0]])
print('Exposición neta:',round(w.sum(),10),'| Bruta:',round(w.abs().sum(),10), '| activos por lado:', int((w>0).sum()))

## 7. Motor de rentabilidad y costes
$F_t=\sum_i w_{i,t}r_{i,t}$. El coste aproximado es $c\sum_i|w_{i,t}-w_{i,t-1}|$, donde c es el coste por nominal negociado. Incluye apertura inicial, no liquidación terminal. Es una aproximación de rotación de pesos objetivo con capital de referencia fijo: no ajusta la deriva entre rebalanceos, préstamo, funding, margen, fiscalidad o ejecuciones. No es un backtest ejecutable.

In [ ]:
print(inspect.getsource(build_factor))

factor_details, net, gross = factor_table(panel, SCORES, COST_BPS, QUANTILE)
print('Ejemplo de factor Value:')
display(factor_details['value'].head(4).round(5))

## 8. Desarrollo y validación fuera de muestra
Fijamos el corte cronológico antes de mirar resultados. No seleccionamos ni cambiamos parámetros usando la validación. Comparamos medias mensuales y t estadístico con errores HAC (Newey-West, 3 retardos) como diagnóstico introductorio. El p-valor no corrige multiplicidad en esta tabla: las conclusiones son exploratorias (la corrección por comparaciones múltiples de P11 se aplica en el E8).

In [ ]:
print(inspect.getsource(summarize))

split=all_months[int(len(all_months)*TRAIN_SHARE)]
dev, val = period_summary(net, split)
print('Corte:',split.date(),'| medias netas salvo mercado (sin costes)')
display(pd.concat({'desarrollo':dev,'validación':val},axis=1).round(2))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Value se mantiene; carry sale negativo sin prima programada</b>

| Factor | Desarrollo: media (bps) | t | Validación: media (bps) | t |
|---|---|---|---|---|
| Size | +74,3 | **3,88** | +23,9 | 0,95 |
| **Value** | **+119,0** | **5,42** | **+127,9** | **4,55** |
| Quality | −11,3 | −0,54 | +23,3 | 0,60 |
| Momentum | +41,4 | 2,14 | +37,3 | 1,16 |
| Low Vol | +27,6 | 1,51 | +4,9 | 0,20 |
| **Carry** | **−36,7** | **−2,21** | **−67,5** | **−2,61** |
| Liquidity | +23,5 | 1,17 | +31,2 | 1,41 |
| Investment | +1,8 | 0,08 | +52,8 | 1,51 |
| Mercado | +65,4 | 2,06 | +55,7 | 1,07 |

- **Value**, con su prima programada, es el único factor con t > 2 en los dos periodos.
- **Size**, también programado, es significativo en desarrollo pero cae a un tercio en validación: una prima real puede no verse en 42 meses.
- **Carry no tiene ninguna prima programada** y sale negativo y "significativo" en los dos periodos. El E5 explica por qué: hereda la prima de value por azar.
- Momentum tiene 66 meses en desarrollo (necesita 12 de historia) y Low Vol, 72 (necesita 6).

</div>

### 8.1 Estabilidad entre periodos
El signo de una prima puede cambiar. Una media positiva no basta para afirmar que existe una prima estructural.

In [ ]:
ax=pd.DataFrame({'Desarrollo':dev.mean_bps,'Validación':val.mean_bps}).plot.bar(figsize=(10,4))
ax.axhline(0,color='grey',lw=1); ax.set(ylabel='Retorno mensual medio (bps)',title=f'Factores por periodo · {DATA_MODE}'); plt.xticks(rotation=30,ha='right'); plt.tight_layout(); plt.show()

### 8.2 Efecto de costes
Comparamos sobre idénticos meses. La reducción de retorno es mecánica por construcción; los costes reales dependen del instrumento y del tamaño.

In [ ]:
paired=gross.loc[gross.index>=split]; after=net.loc[paired.index,SCORES]
comparison=pd.DataFrame({'Bruto':paired.mean()*1e4,'Neto':after.mean()*1e4,
                         'Rotación media':{s: factor_details[s].turnover.loc[paired.index].mean() for s in SCORES}})
display(comparison.round(2))
ax=comparison[['Bruto','Neto']].plot.bar(figsize=(10,4));ax.axhline(0,color='grey',lw=1);ax.set(ylabel='Media mensual (bps)',title=f'Validación: efecto de costes · {DATA_MODE}');plt.xticks(rotation=30,ha='right');plt.tight_layout();plt.show()

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Los costes castigan a los factores que rotan</b>

| Factor | Bruto (bps) | Neto (bps) | Rotación mensual | Coste (bps) |
|---|---|---|---|---|
| Low Vol | 20,8 | **4,9** | **1,06** | **15,9** |
| Momentum | 48,9 | 37,3 | 0,77 | 11,5 |
| Size | 27,0 | 23,9 | 0,21 | 3,1 |
| Value | 130,9 | 127,9 | 0,20 | 3,0 |
| Quality | 24,4 | 23,3 | 0,08 | 1,2 |
| Carry, Liquidity, Investment | — | igual al bruto | 0 | 0 |

- **El coste es la rotación por 15 bps:** Low Vol cambia más del 100 % de su nominal cada mes y pierde tres cuartas partes de su retorno bruto.
- Carry, Liquidity e Investment no rotan nunca, porque sus scores son constantes: tras la apertura, la cartera es siempre la misma. Ya no es una estrategia que responda a información nueva.

</div>

### 8.3 Factores correlacionados
Una matriz de correlaciones calculada sobre meses comunes muestra si las estrategias comparten exposiciones; no prueba causalidad.

In [ ]:
common=net.dropna(); corr=common.corr()
fig,ax=plt.subplots(figsize=(7,6)); im=ax.imshow(corr,vmin=-1,vmax=1,cmap='coolwarm')
ax.set_xticks(range(len(corr)),corr.columns,rotation=45,ha='right');ax.set_yticks(range(len(corr)),corr.index)
for i in range(len(corr)):
    for j in range(len(corr)):
        v=corr.iloc[i,j];ax.text(j,i,f'{v:.2f}',ha='center',va='center',fontsize=8,color='white' if abs(v)>.65 else 'black')
fig.colorbar(im,ax=ax,label='Correlación');ax.set_title(f'Correlación de factores · {len(common)} meses comunes · {DATA_MODE}');ax.grid(False);fig.tight_layout();plt.show()

## 9. Modelos Fama-French: referencia real
El modelo de 3 factores incorpora Mkt-RF, SMB (small minus big) y HML (high minus low book-to-market). El de 5 añade RMW (rentabilidad operativa robusta menos débil) y CMA (inversión conservadora menos agresiva). Momentum se incorpora en otros modelos, no es uno de los cinco oficiales. Los factores de la biblioteca usan carteras específicas ponderadas por valor y ordenaciones cruzadas; nuestros factores simples son proxies, no réplicas.

Fuentes: [Biblioteca Ken French](https://mba.tuck.dartmouth.edu/pages/faculty/Ken.French/data_library.html), [3 factores](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library/f-f_factors.html), [5 factores](https://mba.tuck.dartmouth.edu/pages/faculty/Ken.French/Data_Library/f-f_5_factors_2x3.html).

Descarga opcional manual: selecciona las series mensuales estadounidenses de 3 y 5 factores CSV, descomprime y asigna las rutas arriba. El lector toma solo filas YYYYMM y convierte porcentajes a decimales. Introduce también `STRATEGY_CSV` con retornos mensuales totales en USD, sin mezclar con retornos diarios en euros de P09. La comparación de modelos se realiza sobre los mismos meses. Esta celda no descarga nada; la descarga automática está en el E7.

In [ ]:
print(inspect.getsource(read_ff_monthly))
print(inspect.getsource(fit_model))

if FF3_CSV and FF5_CSV and STRATEGY_CSV:
    f3=read_ff_monthly(FF3_CSV,['Mkt-RF','SMB','HML','RF'])
    f5=read_ff_monthly(FF5_CSV,['Mkt-RF','SMB','HML','RMW','CMA','RF'])
    strategy=pd.read_csv(STRATEGY_CSV,parse_dates=['month']).set_index('month').strategy_return
    assert strategy.index.is_unique
    shared=pd.concat([strategy.rename('strategy'),f3.add_suffix('_3'),f5.add_suffix('_5')],axis=1).dropna()
    for suffix,cols in [('_3',['Mkt-RF','SMB','HML']),('_5',['Mkt-RF','SMB','HML','RMW','CMA'])]:
        model=fit_model(shared.strategy-shared['RF'+suffix],shared[[col+suffix for col in cols]])
        print(suffix, model_summary(model))
else:
    # Demostración artificial de la mecánica de modelos anidados, no FF oficial.
    d=net[['market','size','value','quality','investment']].dropna()
    rng=np.random.default_rng(SEED+1)
    y=.8*d.market+.4*d['size']+.5*d.value+.5*d.quality+.3*d.investment+rng.normal(0,.01,len(d))
    for cols in [['market','size','value'],['market','size','value','quality','investment']]:
        model=fit_model(y,d[cols]); print(len(cols),'proxies sintéticos | alpha (bps):',round(model.params['const']*1e4,2),'| p:',round(model.pvalues['const'],3),'| R²:',round(model.rsquared,3))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Un factor omitido aparece como alpha</b>

La estrategia artificial se construye con cinco factores y **ningún alpha**.

- **Con 3 factores:** alpha de **−28,3 bps** al mes (p = 0,09) y R² de 0,79. El alpha "negativo" es la parte de quality e investment que el modelo omite y que los otros tres factores no recogen.
- **Con los 5 factores:** alpha de −10,9 bps (p = 0,23), compatible con 0, y R² de 0,92. Lo que queda es el ruido añadido (1 % al mes).

Un alpha solo tiene sentido **respecto a un modelo concreto**: añadir factores puede hacerlo desaparecer.

</div>

## 10. Laboratorio de derivados (lección 1.30)
Ejercicio de P&L a vencimiento sobre 1 BTC equivalente, sin financiación de la prima ni costes adicionales. Para un futuro lineal el resultado acumulado es $S_T-F_0$; la liquidación real puede ocurrir antes por margen. Para la call comprada el resultado es $\max(S_T-K,0)-p$; para la put, $\max(K-S_T,0)-p$. Un forward lineal produce el mismo payoff terminal bajo iguales condiciones. En un swap de tipos, el flujo neto del pagador fijo es $N(r_{variable}-r_{fijo})\Delta t$.

In [ ]:
print(inspect.getsource(derivative_pnl))

payoffs=derivative_pnl(np.arange(30000,90001,1000))
ax=payoffs.plot();ax.axhline(0,color='grey');ax.set(xlabel='BTC a vencimiento (EUR/BTC)',ylabel='P&L (EUR)',title='Payoffs hipotéticos sobre 1 BTC');plt.tight_layout();plt.show()
print(f'Swap: flujo neto anual del pagador fijo (variable 4 %, fijo 3 %): {swap_net_flow(1_000_000, .04, .03):,.2f} EUR')
print(f'Basis anualizado simple hipotético (61.500 / 60.000 a 90 días): {annualized_basis(61500, 60000, 90):.2%}')

## 11. Comprobaciones automáticas
Las pruebas de abajo comprueban convenciones, alineación y reconciliación económica. No validan que un factor tenga capacidad predictiva real.

In [ ]:
# 🔎 FASE DE VALIDACIÓN: pesos, scores sin futuro, reconciliación de costes, derivados y lector FF
test=pd.DataFrame({'asset':['A','B','C','D'],'score':[1,2,3,4]})
tw=long_short_weights(test,'score',.25)
assert np.isclose(tw.sum(),0)
assert np.isclose(tw.abs().sum(),2)
assert tw['A']==-1 and tw['D']==1
assert long_short_weights(test.assign(score=1),'score',.25).empty
assert mom.iloc[:12].isna().all().all()
assert lowvol.iloc[:6].isna().all().all()
assert (panel.available_at < panel.month).all()
for details in factor_details.values():
    assert np.allclose(details.gross-details.cost,details.net,equal_nan=True)
    assert (details.cost>=0).all()
assert derivative_pnl([60000]).iloc[0].tolist()==[0,-2000,-2000]
assert derivative_pnl([62000]).iloc[0]['Call comprada']==0
assert derivative_pnl([58000]).iloc[0]['Put comprada']==0
# Prueba de lector oficial con formato diminuto ficticio, sin red.
from tempfile import TemporaryDirectory
with TemporaryDirectory() as tmp:
    fake=Path(tmp)/'ff.csv';fake.write_text('cabecera\n,Mkt-RF,SMB,HML,RF\n202001,1,-2,3,0.1\n2020,10,10,10,1\n')
    parsed=read_ff_monthly(fake,['Mkt-RF','SMB','HML','RF'])
    assert len(parsed)==1 and np.isclose(parsed.iloc[0].SMB,-.02)
print('Comprobaciones superadas (incluida reconciliación de los ocho factores y lector CSV ficticio).')

## 12. Ejercicios
1. **Ranking:** implementa los pesos long-short sin llamar a `long_short_weights`. Usa cuatro activos con scores 1,2,3,4 y retornos 1%,2%,3%,4%. Comprueba el neto 0 y el bruto 2 de los pesos.
2. **Momentum:** calcula a mano un score de 11 retornos y comprueba qué ocurre si modificas el retorno del mes objetivo. Explica por qué no debería cambiar el score.
3. **Value y Size:** compara terciles y quintiles solo en desarrollo; registra la hipótesis antes de examinar validación. ¿Qué ocurre con la rotación?
4. **Costes:** calcula el coste máximo por unidad de nominal para el que la media neta de Value sería positiva en desarrollo (pista: media bruta / rotación media). No optimices con validación.
5. **Factores:** compara la correlación Quality/Market y Value/Size. ¿Es suficiente para identificar factores independientes?
6. **Derivados:** calcula a mano el P&L de call, put, futuro y swap para tres escenarios. Diseña un score carry para perpetuos usando exclusivamente funding conocido en la fecha de formación; distingue pago observado y estimado.
7. **Datos reales:** importa un panel punto-en-tiempo de acciones o un universo cripto adaptado; en cripto sustituye Value y Quality por proxies justificados o elimínalos. No rellenes características inexistentes. Para los modelos FF, utiliza las series oficiales y una estrategia en la misma moneda y frecuencia.
8. **Informe:** formula una hipótesis sobre una prima de riesgo, fija universo, horizonte, costes y criterio de rechazo. Informa tamaño muestral, media, estabilidad, corrección por múltiples comparaciones y limitaciones.

### E1 · Pesos long-short a mano

In [ ]:
print(inspect.getsource(manual_long_short))

e1 = manual_long_short({'A': 1, 'B': 2, 'C': 3, 'D': 4}, {'A': .01, 'B': .02, 'C': .03, 'D': .04}, QUANTILE)
print(e1)
# Con el panel: los pesos manuales coinciden con long_short_weights en el mes de ejemplo.
manual = manual_long_short(dict(zip(sample.asset, sample.value)), dict(zip(sample.asset, sample.asset_return)), QUANTILE)
assert all(np.isclose(manual['weights'][a], w.get(a, 0)) for a in sample.asset)
print(f"Mes {sample_month.date()}: pesos manuales = long_short_weights en los {len(sample)} activos")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E1 · Uno largo, uno corto</b>

- Con 4 activos y un cuantil del 30 %, k = ⌊4 · 0,30⌋ = **1**: compramos D (score 4) con peso +1 y vendemos A (score 1) con peso −1. B y C quedan con peso 0.
- **Neto 0, bruto 2**, y el retorno del factor es 4 % − 1 % = **3 %**.
- Con los 60 activos del mes de ejemplo, los pesos manuales coinciden exactamente con `long_short_weights` (18 por lado, ±1/18).

</div>

### E2 · Momentum 12-2 a mano

In [ ]:
print(inspect.getsource(momentum_12_2))

asset, target = 'A000', all_months[30]
r = panel[panel.asset == asset].set_index('month').asset_return
window = r.loc[target - pd.DateOffset(months=12): target - pd.DateOffset(months=2)]
print(f"{asset} · mes objetivo {target.date()} · ventana {window.index.min().date()} a {window.index.max().date()} ({len(window)} retornos)")
print(f"A mano: {np.prod(1 + window) - 1:.6f} | función: {momentum_12_2(r, target):.6f} | panel: {panel.set_index(['month','asset']).momentum[(target, asset)]:.6f}")
for label, changed in [('mes objetivo t', target), ('t − 1', all_months[29]), ('t − 2', all_months[28])]:
    altered = r.copy(); altered[changed] = .5
    print(f"Retorno de {label} cambiado a +50 % → momentum {momentum_12_2(altered, target):.6f}")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E2 · El score no puede depender del retorno que quiere predecir</b>

- A000, julio de 2017: los 11 retornos de **julio de 2016 a mayo de 2017** dan un momentum de **+9,37 %**. El cálculo a mano, la función y el panel coinciden.
- Si cambiamos el retorno **del mes objetivo** (o el de t − 1) a +50 %, el score **no cambia**. Si cambiamos el de t − 2, pasa a +57,4 %: ese mes sí está en la ventana.
- **Por qué no debe cambiar:** el score se usa para decidir la cartera **al empezar** el mes t, cuando su retorno aún no existe. Si dependiera de él, el backtest usaría el futuro. Saltar t − 1 es una convención (evita la reversión de corto plazo y da margen para conocer el cierre), no una necesidad lógica.

</div>

### E3 · Terciles frente a quintiles, solo en desarrollo
**Hipótesis registrada antes de mirar:** con quintiles (12 activos por lado) el spread será **mayor** que con terciles (20 por lado), porque los extremos están más separados en la característica. La rotación también será **mayor**, porque una cartera más pequeña cambia más cuando un activo entra o sale.

In [ ]:
print(inspect.getsource(quantile_comparison))

display(quantile_comparison(panel, split, ('value', 'size'), (1/3, QUANTILE, .2), COST_BPS).round(3))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E3 · El spread crece con los extremos; la rotación, solo en value</b>

| Factor | Cuantil | Por lado | Bruto (bps) | Neto (bps) | Rotación (sin apertura) |
|---|---|---|---|---|---|
| Value | Terciles | 20 | 119,2 | 116,4 | 0,168 |
| Value | 30 % | 18 | 122,1 | 119,0 | 0,186 |
| Value | Quintiles | 12 | **144,2** | **139,4** | **0,299** |
| Size | Terciles | 20 | 67,8 | 64,4 | 0,203 |
| Size | 30 % | 18 | 78,1 | 74,3 | 0,235 |
| Size | Quintiles | 12 | **98,7** | **95,4** | 0,199 |

- **La primera parte de la hipótesis se cumple:** los quintiles dan un spread mayor en los dos factores (+25 bps en value y +31 en size).
- **La segunda, solo en value:** su rotación casi se duplica con quintiles. En size se queda igual (0,199 frente a 0,203).
- Con 15 bps por lado, el coste extra de los quintiles en value (unos 2 bps) es muy inferior al spread que se gana. Pero estas cifras son de desarrollo: no se elige el cuantil mirando la validación.

</div>

### E4 · Coste de equilibrio

In [ ]:
print(inspect.getsource(breakeven_cost_bps))

e4 = pd.DataFrame({s: {'bruto_bps': factor_details[s].gross[factor_details[s].index < split].mean() * 1e4,
                       'rotacion_media': factor_details[s].turnover[(factor_details[s].index < split) & factor_details[s].gross.notna()].mean(),
                       'coste_equilibrio_bps': breakeven_cost_bps(factor_details[s], split)}
                   for s in ['value', 'size', 'momentum', 'low_vol']}).T
display(e4.round(3))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E4 · Value aguantaría costes enormes; Low Vol, no</b>

Coste por lado y por unidad de nominal que deja la media neta de desarrollo en 0 (= media bruta / rotación media):

| Factor | Media bruta (bps) | Rotación media | Coste de equilibrio |
|---|---|---|---|
| **Value** | 122,1 | 0,209 | **583 bps** |
| Size | 78,1 | 0,258 | 303 bps |
| Momentum | 52,7 | 0,753 | 70 bps |
| Low Vol | 43,4 | 1,051 | **41 bps** |

- Value seguiría siendo positivo con costes de hasta un **5,8 %** por lado, porque rota poco y su spread es grande. En la realidad no hay primas así: es la prima programada de la simulación.
- **Low Vol solo aguanta 41 bps:** rota más del 100 % del nominal cada mes.
- La rotación media cuenta solo los meses con factor (incluida la apertura, 2,0 el primer mes). Calculado solo con desarrollo, como exige el enunciado.

</div>

### E5 · ¿Correlación baja = factores independientes?

In [ ]:
print(inspect.getsource(spanning_alpha))

print(f"Correlación de retornos ({len(common)} meses comunes): Quality/Market {corr.loc['quality','market']:.3f} | "
      f"Value/Size {corr.loc['value','size']:.3f} | Carry/Value {corr.loc['carry','value']:.3f}")
first = panel[panel.month == all_months[0]].set_index('asset')
print(f"Correlación transversal de scores en {all_months[0].date()}: carry–value {first.carry.corr(first.value):.3f} | carry–size {first.carry.corr(first['size']):.3f}")
print('Carry explicado por los factores value y size (desarrollo):', {k: round(v, 3) for k, v in spanning_alpha(net[net.index < split], 'carry', ['value', 'size']).items()})
# Prima de carry implícita en su inclinación hacia las características con prima (solo posible en la simulación:
# recuperamos las variables latentes con las que se generaron los retornos).
latent = panel.assign(lat_value=(np.log(panel.book_to_market) + .5) / .35, lat_size=np.log(panel.market_cap) - 20)
implied = pd.Series({m: (lambda g, w: (w * (.004 * g.lat_value - .002 * g.lat_size).loc[w.index]).sum())(g.set_index('asset'), long_short_weights(g, 'carry', QUANTILE))
                     for m, g in latent.groupby('month')})
print(f"Prima implícita de carry: desarrollo {implied[implied.index < split].mean()*1e4:.1f} bps | validación {implied[implied.index >= split].mean()*1e4:.1f} bps "
      f"(observada: {factor_details['carry'].gross[factor_details['carry'].index < split].mean()*1e4:.1f} y {factor_details['carry'].gross[factor_details['carry'].index >= split].mean()*1e4:.1f})")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E5 · Una correlación baja no demuestra independencia</b>

- **Quality/Market: 0,09** y **Value/Size: 0,27** (108 meses comunes). Las dos son bajas, pero eso no basta para hablar de factores independientes.
- **El contraejemplo es carry:**
  - Su correlación de retornos con value es solo **−0,13**, y los factores value y size explican apenas el **2 %** de su varianza (R²).
  - Pero sus scores están correlacionados por azar con los de value al empezar (**−0,25**): la cartera de carry compra activos con poco book-to-market.
  - La prima que "hereda" por esa inclinación es de **−33 bps** en desarrollo, prácticamente toda la media observada (−36). En validación, −35 de −68: el resto es ruido.
- **Lección:** la correlación mide si los retornos se mueven juntos mes a mes, no si comparten la **fuente de su media**. Para identificar factores independientes hacen falta ordenaciones dobles (por ejemplo, carry dentro de cada tercil de value), regresiones de spanning sobre las características y estabilidad en varios periodos y universos.

</div>

### E6 · Derivados a mano y carry de perpetuos

In [ ]:
print(inspect.getsource(swap_net_flow))
print(inspect.getsource(perp_carry_score))

scenarios = [45000, 60000, 75000]
by_hand = pd.DataFrame({'Futuro long': [s - 60000 for s in scenarios],
                        'Call comprada': [max(s - 60000, 0) - 2000 for s in scenarios],
                        'Put comprada': [max(60000 - s, 0) - 2000 for s in scenarios]}, index=scenarios)
assert by_hand.equals(derivative_pnl(scenarios).astype(int))
display(by_hand)
print('Swap 1.000.000 EUR, fijo 4 %, un año:', {f'variable {f:.0%}': round(swap_net_flow(1_000_000, f, .04), 2) for f in (.03, .04, .05)})

# Score carry de un perpetuo: 30 liquidaciones artificiales cada 8 h; la última es el "próximo funding" anunciado.
times = pd.date_range('2026-03-01', periods=30, freq='8h', tz='UTC')
funding = pd.DataFrame({'funding_time': times, 'rate': np.r_[np.full(29, .0001), .0012], 'kind': ['observado'] * 29 + ['estimado']})
funding.loc[28, 'rate'] = .0040  # se liquida justo en la hora de formación: todavía no se conoce
formation = times[28]
display(funding.tail(4))
print('Score en la formación', formation, '→', perp_carry_score(funding, formation, 21))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E6 · Payoffs y un carry sin mirar el futuro</b>

**P&L a vencimiento** (1 BTC, strike y precio del futuro de 60.000 €, prima de 2.000 €):

| BTC a vencimiento | Futuro long | Call comprada | Put comprada |
|---|---|---|---|
| 45.000 € | −15.000 € | −2.000 € | **+13.000 €** |
| 60.000 € | 0 € | −2.000 € | −2.000 € |
| 75.000 € | **+15.000 €** | **+13.000 €** | −2.000 € |

- El futuro es lineal. Las opciones limitan la pérdida a la prima, a cambio de pagarla siempre.
- **Swap** de 1.000.000 € con fijo del 4 % durante un año: el pagador fijo **paga 10.000 €** si el variable es 3 %, nada si es 4 % y **cobra 10.000 €** si es 5 %.

**Score carry de un perpetuo** (`perp_carry_score`):

- Usa solo fundings **observados**, ya liquidados **antes** de la hora de formación. Los 21 últimos (una semana, con 3 liquidaciones diarias) dan un funding medio del **0,01 %** y un carry para el lado largo de **−10,95 % anual**: los largos pagan.
- El funding del 0,40 % que se liquida **justo a la hora de formación** no entra: todavía no se conoce.
- El **estimado** (0,12 %, el próximo funding anunciado) se informa aparte y no entra en el score: puede cambiar antes de liquidarse.

</div>

### E7 · Datos reales: Fama-French oficial y un universo cripto
Cambia `LIVE_E7` a `True` para descargar los ficheros oficiales de Ken French y los precios diarios de Yahoo Finance (requiere `requests` y `yfinance`). Si alguna descarga falla, la celda se detiene con el error.

- **Fama-French:** SPY (control: debería ser casi todo mercado) y GLD (no es una acción), en USD mensual, como los factores.
- **Cripto:** 20 criptomonedas. Value, quality, carry, investment y size se **eliminan**: no hay book-to-market, ROA, crecimiento de activos ni una capitalización fiable en Yahoo. Se conservan Momentum 12-2, Low Vol y Liquidity (iliquidez de Amihud del mes anterior con el volumen en USD). RF: el de Ken French.

In [ ]:
print(inspect.getsource(crypto_monthly_panel))

LIVE_E7 = False
CRYPTO_SCORES = ['momentum', 'low_vol', 'liquidity']

if LIVE_E7:
    from tempfile import TemporaryDirectory
    with TemporaryDirectory() as tmp:
        f3 = read_ff_monthly(download_ff('ff3', tmp), FF_FILES['ff3'][1])
        f5 = read_ff_monthly(download_ff('ff5', tmp), FF_FILES['ff5'][1])
    raw = download_daily([f'{t}-USD' for t in CRYPTO_TICKERS] + ['SPY', 'GLD'], '2014-01-01', '2026-11-01')
    print(f"Descarga {pd.Timestamp.now(tz='UTC'):%Y-%m-%d %H:%M} UTC | Ken French {f3.index.min():%Y-%m} a {f3.index.max():%Y-%m} | "
          f"Yahoo {raw.index.min().date()} a {raw.index.max().date()}")

    # 1) Modelos FF sobre estrategias en USD mensual.
    ff_rows = {}
    for name in ('SPY', 'GLD'):
        r = monthly_returns(raw['Close'][name])
        for model, f, cols in [('FF3', f3, ['Mkt-RF', 'SMB', 'HML']), ('FF5', f5, ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA'])]:
            fit = fit_model(r - f.RF, f[cols])
            ff_rows[(name, model)] = {**model_summary(fit), 'beta_mercado': fit.params['Mkt-RF']}
    display(pd.DataFrame(ff_rows).T.round(4))

    # 2) Factores en un universo cripto.
    close = raw['Close'][[f'{t}-USD' for t in CRYPTO_TICKERS]].rename(columns=lambda c: c[:-4])
    volume = raw['Volume'][[f'{t}-USD' for t in CRYPTO_TICKERS]].rename(columns=lambda c: c[:-4])
    crypto, *_ = add_scores(validate_panel(crypto_monthly_panel(close, volume, f3.RF)))
    crypto = crypto[crypto.month >= '2018-12-01']  # primer mes con los tres scores (Yahoo empieza en nov. de 2017)
    c_details, c_net, _ = factor_table(crypto, CRYPTO_SCORES, COST_BPS, QUANTILE)
    c_split = c_net.index[int(len(c_net) * TRAIN_SHARE)]
    c_dev, c_val = period_summary(c_net, c_split)
    print(f"Cripto: {c_net.index.min().date()} a {c_net.index.max().date()} ({len(c_net)} meses) | corte {c_split.date()} | "
          f"activos por mes: {crypto.groupby('month').asset_return.count().min()}–{crypto.groupby('month').asset_return.count().max()}")
    display(pd.concat({'desarrollo': c_dev, 'validación': c_val}, axis=1).round(2))
    display(multiple_testing(c_dev.p_hac.drop('market')).round(3))
    print('Rotación media:', {s: round(c_details[s].turnover.mean(), 3) for s in CRYPTO_SCORES})
    mom_fit = fit_model(c_net.momentum, f5[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']])
    print('Momentum cripto sobre FF5:', {k: round(v, 4) for k, v in model_summary(mom_fit).items()})
else:
    print("LIVE_E7 = False: no se descarga nada. Resultados documentados en la conclusión de abajo.")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E7 · Datos reales (Ken French y Yahoo Finance, descarga del 20-11-2026)</b>

**Fama-French** (ficheros oficiales, creados con la base CRSP de 2026-09; 152 meses, de febrero de 2014 a septiembre de 2026, en USD):

| Estrategia | Modelo | Alpha mensual | p | R² | Beta de mercado |
|---|---|---|---|---|---|
| SPY | FF3 | +0,6 bps | 0,80 | **0,996** | 0,98 |
| SPY | FF5 | −0,1 bps | 0,95 | 0,996 | 0,99 |
| GLD | FF3 | +53,2 bps | 0,14 | **0,03** | 0,12 |
| GLD | FF5 | +50,5 bps | 0,17 | 0,06 | 0,13 |

- **SPY es el control:** el mercado lo explica casi todo (R² 0,996) y el alpha es 0. El lector y la regresión funcionan.
- **GLD no se explica con factores de acciones** (R² del 3–6 %). Su "alpha" de 50 bps al mes no es significativo y solo dice que el oro tiene su propia fuente de retorno.

**Universo cripto** (20 monedas, de diciembre de 2018 a septiembre de 2026: 94 meses, de 16 a 20 activos por mes, 6 por lado; corte en enero de 2024; 15 bps por lado):

| Factor | Desarrollo: media (bps) | t | Validación: media (bps) | t | Rotación mensual |
|---|---|---|---|---|---|
| Momentum 12-2 | +114,9 | 0,53 | +175,9 | 0,87 | 0,83 |
| Low Vol | −386,6 | −1,54 | −119,7 | −0,34 | 0,92 |
| Liquidity | **+938,9** | **2,13** | +129,3 | 0,36 | 0,37 |

- **Ningún factor sobrevive.** Liquidity tiene p = 0,03 en desarrollo, pero con tres pruebas, Bonferroni (0,10) y BH (0,10) no la rechazan, y en validación cae a +129 bps con t = 0,36.
- La volatilidad de estos factores es enorme: del 58 al 112 % anual. Con 6 activos por lado y meses como diciembre de 2017 (TRX +1.894 %), las medias no son fiables.
- El momentum cripto apenas tiene relación con los factores de acciones (R² 0,06 sobre FF5).
- El mercado cripto equiponderado no tiene t en desarrollo: AVAX no tiene retorno en agosto y septiembre de 2020 (antes de su lanzamiento), el mes queda desconocido y el HAC no se calcula con huecos.

**Limitaciones:** sesgo de supervivencia (el universo se eligió hoy, entre monedas que siguen cotizando; faltan las que desaparecieron, como LUNA); precios y volúmenes agregados de Yahoo, no ejecutables; la fecha de inicio es la de los datos de Yahoo, no la del lanzamiento.

</div>

### E8 · Informe: una hipótesis registrada antes de la validación
**Hipótesis (fijada solo con el desarrollo):** *el factor Value long-short, con un 30 % por lado y 15 bps de coste por lado, tiene una media neta mensual positiva en validación.*

- **Universo:** los 60 activos del panel; **horizonte:** mensual, de julio de 2021 a diciembre de 2024 (42 meses).
- **Criterio de rechazo:** se rechaza la hipótesis si la media de validación no es positiva, o si su p-valor HAC ajustado por BH (con los 8 factores contrastados) no es < 0,05.

In [ ]:
print(inspect.getsource(multiple_testing))

report = pd.DataFrame({'n_val': val.n, 'media_dev_bps': dev.mean_bps, 'media_val_bps': val.mean_bps,
                       'mismo_signo': np.sign(dev.mean_bps) == np.sign(val.mean_bps)}).drop('market')
display(report.join(multiple_testing(val.p_hac.drop('market'))).round(4))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E8 · Value se mantiene; carry, también, y es un falso descubrimiento</b>

| Factor | Media dev. (bps) | Media val. (bps) | p (val.) | Bonferroni | BH |
|---|---|---|---|---|---|
| **Value** | +119,0 | **+127,9** | 0,0000 | **rechaza H₀** | **rechaza H₀** |
| **Carry** | −36,7 | −67,5 | 0,009 | 0,072 (no) | **0,036 (rechaza)** |
| Investment | +1,8 | +52,8 | 0,13 | no | no |
| Liquidity | +23,5 | +31,2 | 0,16 | no | no |
| Momentum | +41,4 | +37,3 | 0,25 | no | no |
| Size | +74,3 | +23,9 | 0,34 | no | no |
| Quality | −11,3 | +23,3 | 0,55 | no | no |
| Low Vol | +27,6 | +4,9 | 0,84 | no | no |

- **La hipótesis se mantiene:** Value tiene una media de **+127,9 bps** al mes en validación (n = 42), del mismo signo que en desarrollo, y sobrevive a Bonferroni y a BH.
- **Pero BH también "descubre" carry,** que no tiene ninguna prima programada (E5). La corrección por multiplicidad controla el azar de muchas pruebas, no un factor que imita a otro: la media de carry es real dentro de esta muestra, pero viene de value.
- Size, que **sí tiene prima programada**, no la detecta en validación: con 42 meses, una prima de unos 25 bps al mes está dentro del ruido.
- **Limitaciones:** pesos equiponderados y univariantes; costes fijos de 15 bps; sin préstamo de títulos, funding, impuestos ni capacidad. Las primas de esta simulación son programadas: con datos reales (E7), ninguna sobrevivió.

</div>

## 13. Exportación e integración
El siguiente bloque es opcional. Guarda retornos netos mensuales de los factores en formato ancho, compatibles con futuras regresiones (no confundir con retornos diarios P12). Etiqueta origen y frecuencia. El paso siguiente P17 integra la investigación sobre datos reales.

In [ ]:
EXPORT=False
if EXPORT:
    out=Path('quantlab_exports');out.mkdir(exist_ok=True)
    net.rename_axis('month').to_csv(out/f'p16_factors_monthly_{DATA_MODE}.csv')
    pd.concat(factor_details,names=['factor']).to_csv(out/f'p16_factor_details_{DATA_MODE}.csv')
    import json
    (out/f'p16_metadata_{DATA_MODE}.json').write_text(json.dumps({'mode':DATA_MODE,'frequency':'monthly','return_unit':'decimal','quantile':QUANTILE,'cost_bps':COST_BPS,'split':str(split),'seed':SEED},indent=2))

## 14. Conclusiones

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIONES DE LA PRÁCTICA P16</b>

**¿Qué factores conservan el signo entre desarrollo y validación?** En la simulación, todos menos Quality: Size, Value, Momentum, Low Vol, Liquidity e Investment siguen positivos y Carry sigue negativo. Pero conservar el signo no es lo mismo que ser significativo: solo Value y Carry tienen t > 2 en validación, y Carry no tiene ninguna prima programada (la hereda de value por azar, E5).

**¿Cuál cambia más después de costes? ¿Cuánta rotación lo explica?** Low Vol: rota 1,06 veces su nominal al mes y los 15 bps por lado le quitan 15,9 de sus 20,8 bps brutos. Momentum pierde 11,5 bps con una rotación de 0,77. Los factores con scores constantes no pagan nada después de la apertura.

**¿Por qué los factores de este cuaderno no son SMB, HML, RMW o CMA oficiales?** Los oficiales usan ordenaciones dobles (tamaño × característica) con cortes de NYSE, carteras ponderadas por capitalización, rebalanceo anual en junio con datos contables del año fiscal anterior y todo el universo CRSP con sus bajas. Los nuestros son ordenaciones univariantes, equiponderadas y mensuales sobre 60 activos artificiales.

**¿Qué información real necesitarías para construirlos de forma rigurosa?** Un panel point-in-time de todo el universo (incluidas las empresas que dejaron de cotizar y su retorno de exclusión), fundamentales con su fecha real de publicación, capitalizaciones, costes y restricciones de préstamo, y los factores oficiales para comparar. En cripto, además, un universo sin sesgo de supervivencia y precios ejecutables (E7).

**Límite:** el resultado sintético solo comprueba la mecánica. No constituye evidencia de una prima de riesgo real ni de rentabilidad futura. Con datos reales, ninguno de los tres factores cripto sobrevivió a la validación.

📄 Resumen en `docs/conclusiones/P16_factor_construction.md`

</div>